# 🤖 Scikit-learn — Machine Learning
## Python Ecosystem Tutorial Series — Module 2 of 18

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

| | |
|---|---|
| **Library** | 🤖 Scikit-learn |
| **Domain** | Machine Learning |
| **Dataset** | Iris flowers |
| **Module** | 2 of 18 |

**What you will learn:**

1. What Scikit-learn is and why it exists
2. Core concepts and data structures
3. Hands-on code with real data
4. Visualisations and interpretation
5. When to use it and alternatives

```bash
# Install required libraries
pip install scikit-learn
```

## Quick Reference Card

| Code | What it does |
|------|--------------|
| `model.fit(X, y)` | Train the model |
| `model.predict(X)` | Make predictions |
| `Pipeline([...])` | Chain steps safely |
| `cross_val_score()` | Reliable accuracy |
| `GridSearchCV()` | Tune hyperparameters |

# 2. 🤖 Scikit-learn — Machine Learning
> **Python + Scikit-learn = Machine Learning**

Scikit-learn is the go-to library for classical ML. Consistent API:
`fit()` → `predict()` → `score()` works for every model.

**Key concepts:** classification, train/test split, cross-validation, pipelines, hyperparameter tuning

In [ ]:
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve)
import matplotlib.pyplot as plt
import numpy as np

# ── Load Iris dataset ─────────────────────────────────────────────────────────
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names  = iris.target_names

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Classes: {list(target_names)}")
print(f"Features: {feature_names}")

# ── Train / test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      random_state=42, stratify=y)
print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# ── Build a Pipeline (preprocessing + model) ──────────────────────────────────
# Pipeline chains steps: StandardScaler → RandomForest
pipeline = Pipeline([
    ("scaler", StandardScaler()),          # Step 1: normalise features
    ("model",  RandomForestClassifier(n_estimators=100, random_state=42))  # Step 2: classify
])

pipeline.fit(X_train, y_train)
acc = pipeline.score(X_test, y_test)
print(f"Random Forest accuracy: {acc:.3f}")
print()
print(classification_report(y_test, pipeline.predict(X_test), target_names=target_names))

# ── Compare 4 models ──────────────────────────────────────────────────────────
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM":                 SVC(probability=True),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    pipe = Pipeline([("sc", StandardScaler()), ("m", model)])
    cv_scores = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
    results[name] = cv_scores
    print(f"{name:25s}: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

In [ ]:
# ── Visualisation: confusion matrix + feature importance ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1: Confusion matrix
cm = confusion_matrix(y_test, pipeline.predict(X_test))
im = axes[0].imshow(cm, cmap="Blues")
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, cm[i,j], ha="center", va="center",
                     fontweight="bold", fontsize=14,
                     color="white" if cm[i,j]>cm.max()/2 else "black")
axes[0].set_xticks(range(3)); axes[0].set_yticks(range(3))
axes[0].set_xticklabels(target_names, rotation=30)
axes[0].set_yticklabels(target_names)
axes[0].set_title("Confusion Matrix", fontweight="bold")
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Panel 2: Feature importances
rf = pipeline.named_steps["model"]
importances = rf.feature_importances_
axes[1].barh(feature_names, importances, color=["#3498DB","#E74C3C","#27AE60","#F1C40F"],
              alpha=0.85, edgecolor="white")
axes[1].set_title("Feature Importances", fontweight="bold")
axes[1].set_xlabel("Importance")
for i, v in enumerate(importances):
    axes[1].text(v+0.001, i, f"{v:.3f}", va="center", fontsize=9)

# Panel 3: Model comparison
names = list(results.keys())
means = [results[n].mean() for n in names]
stds  = [results[n].std()  for n in names]
cols  = ["#3498DB","#27AE60","#E74C3C","#F1C40F"]
bars = axes[2].barh(names, means, xerr=stds, color=cols, alpha=0.85,
                     edgecolor="white", capsize=4)
axes[2].set_xlim(0.9, 1.01)
axes[2].set_title("Model Comparison (5-fold CV)", fontweight="bold")
axes[2].set_xlabel("Accuracy")
for bar, v in zip(bars, means):
    axes[2].text(v+0.001, bar.get_y()+bar.get_height()/2,
                 f"{v:.3f}", va="center", fontsize=9)

plt.suptitle("Scikit-learn — Iris Classification", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("sklearn_iris.png", dpi=120, bbox_inches="tight")
plt.show()

## Deep Dive: Scikit-learn

### The Unified API
The same interface works for every model:
```python
model.fit(X_train, y_train)    # train
model.predict(X_test)           # predict
model.score(X_test, y_test)     # evaluate
```
Logistic Regression, Random Forest, SVM, k-NN — all identical syntax.

### The Iris Dataset
150 flowers, 3 species, 4 measurements: sepal length/width + petal length/width. The gold standard beginner classification dataset — clean, balanced, small enough to run instantly.

### Why Use a Pipeline?
```python
Pipeline([("scaler", StandardScaler()), ("model", RandomForest())])
```
Chains preprocessing + model into one object. Critical benefit: prevents **data leakage** — the scaler learns statistics only from training data, never from test data.

### Cross-Validation Explained
5-fold CV is better than a single train/test split:
1. Split data into 5 chunks
2. Train on 4 chunks, test on 1 (5 times, rotating the test chunk)
3. Report: mean accuracy +/- std

This gives a reliable estimate of how the model generalises to new data.

### Feature Importances
Petal measurements (length: ~46%, width: ~42%) dominate sepal measurements (~12% combined). This makes biological sense — petal morphology is more species-specific.


## ✅ Key Takeaways — 🤖 Scikit-learn

1. Scikit-learn's unified API makes trying different models trivial
2. Always use Pipeline to prevent data leakage in preprocessing
3. 5-fold cross-validation gives more reliable accuracy than one split
4. Feature importances reveal what your model actually learned

---
*Next: Continue to Module 3 of 18 in the Python Ecosystem Tutorial Series*  
*Portfolio: [hgoelgithub.github.io](https://hgoelgithub.github.io)*